In [2]:
from __future__ import annotations

import argparse
import ast
import math
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D

In [3]:
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "grid.linewidth": 0.7,
})

PALETTE = {
    "blue": "#345995",
    "cyan": "#03CEA4",
    "orange": "#FB4D3D",
    "yellow": "#F5B700",
    "purple": "#7B2CBF",
    "gray": "#6C757D",
    "light_gray": "#F3F5F7",
    "dark": "#1F2933",
    "green": "#2E7D32",
    "red": "#B00020",
    "cream": "#FFF8E7",
}

MODEL_ORDER = ["YandexGPT Lite", "YandexGPT 5 Pro", "YandexGPT 5.1"]
LANG_ORDER = ["EN", "RU"]
SCENARIO_ORDER = ["none", "baseline", "demographic_parity", "auditor", "soft_auditor", "anonymized"]

In [ ]:
from __future__ import annotations

import argparse
import ast
import math
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "grid.linewidth": 0.7,
})

PALETTE = {
    "blue": "#345995",
    "cyan": "#03CEA4",
    "orange": "#FB4D3D",
    "yellow": "#F5B700",
    "purple": "#7B2CBF",
    "gray": "#6C757D",
    "light_gray": "#F3F5F7",
    "dark": "#1F2933",
    "green": "#2E7D32",
    "red": "#B00020",
    "cream": "#FFF8E7",
}

MODEL_ORDER = ["YandexGPT Lite", "YandexGPT 5 Pro", "YandexGPT 5.1"]
LANG_ORDER = ["EN", "RU"]
SCENARIO_ORDER = ["none", "baseline", "demographic_parity", "auditor", "soft_auditor", "anonymized"]

def read_csv_safe(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="utf-8-sig")


def model_from_any(value: object) -> Optional[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).lower()
    if "5_1" in s or "5.1" in s or "5-1" in s:
        return "YandexGPT 5.1"
    if "5_pro" in s or "5-pro" in s or "5 pro" in s:
        return "YandexGPT 5 Pro"
    if "api" in s or "lite" in s:
        return "YandexGPT Lite"
    return str(value)


def model_from_filename(path: Path) -> str:
    return model_from_any(path.name) or path.stem


def normalize_language(x: object) -> str:
    s = str(x).lower()
    if s in {"en", "eng", "english"} or "_en" in s:
        return "EN"
    if s in {"ru", "rus", "russian"} or "_ru" in s:
        return "RU"
    return str(x).upper()


def add_common_columns(df: pd.DataFrame, source_path: Path, scenario: Optional[str] = None) -> pd.DataFrame:
    df = df.copy()
    df["source_file"] = source_path.name

    if "model" in df.columns:
        df["model_display"] = df["model"].map(model_from_any)
    elif "model_key" in df.columns:
        df["model_display"] = df["model_key"].map(model_from_any)
    elif "provider_model" in df.columns:
        df["model_display"] = df["provider_model"].map(model_from_any)
    else:
        df["model_display"] = model_from_filename(source_path)

    # Если модель не распознана из столбца, пробуем из имени файла
    df["model_display"] = df["model_display"].fillna(model_from_filename(source_path))

    if "language" not in df.columns:
        if "_en_" in source_path.name or source_path.name.endswith("_en_results.csv"):
            df["language"] = "EN"
        elif "_ru_" in source_path.name or source_path.name.endswith("_ru_results.csv"):
            df["language"] = "RU"
        else:
            df["language"] = "unknown"
    df["language"] = df["language"].map(normalize_language)

    if scenario is not None:
        df["abm_scale"] = scenario
    return df


def parse_dict_cell(x: object) -> Dict[str, object]:
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    s = str(x).strip()
    if not (s.startswith("{") and s.endswith("}")):
        return {}
    try:
        return ast.literal_eval(s)
    except Exception:
        return {}


def expand_dict_column(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    if col not in df.columns:
        return df
    parsed = df[col].map(parse_dict_cell)
    keys = sorted({k for d in parsed for k in d.keys()})
    for k in keys:
        df[f"{prefix}_{k}"] = parsed.map(lambda d: d.get(k, np.nan))
    return df


def load_many(results_dir: Path, patterns: Iterable[str], exclude_raw: bool = False, scenario: Optional[str] = None) -> pd.DataFrame:
    paths: List[Path] = []
    for pattern in patterns:
        paths.extend(sorted(results_dir.glob(pattern)))
    if exclude_raw:
        paths = [p for p in paths if "raw" not in p.name.lower()]
    frames = []
    for p in paths:
        try:
            df = read_csv_safe(p)
            frames.append(add_common_columns(df, p, scenario=scenario))
        except Exception as e:
            print(f"[WARN] Не удалось прочитать {p}: {e}")
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)


def to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def first_existing(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for n in names:
        if n in df.columns:
            return n
    return None


def load_all(results_dir: Path) -> Dict[str, pd.DataFrame]:
    expressed = load_many(results_dir, ["*expressed_bias_results.csv"], exclude_raw=True)
    encoded = load_many(results_dir, ["*encoded_results.csv"], exclude_raw=True)
    weat = load_many(results_dir, ["*weat_results.csv"], exclude_raw=True)
    jobfair = load_many(results_dir, ["*jobfair_results.csv"], exclude_raw=True)
    jobfair_raw = load_many(results_dir, ["1_*jobfair_raw_*_results.csv", "*jobfair_raw_*_results.csv"], exclude_raw=False)
    abm50 = load_many(results_dir, ["*abm_n50_hq15_results.csv"], exclude_raw=True, scenario="50/15")
    abm100 = load_many(results_dir, ["*abm_n100_hq30_results.csv"], exclude_raw=True, scenario="100/30")

    for df in [jobfair]:
        for col, pref in [("level_bias", "level_bias"), ("spread_bias", "spread_bias"),
                          ("level_test", "level_bias"), ("spread_test", "spread_bias")]:
            df = expand_dict_column(df, col, pref)
        if len(df):
            jobfair = df

    return {
        "expressed": expressed,
        "encoded": encoded,
        "weat": weat,
        "jobfair": jobfair,
        "jobfair_raw": jobfair_raw,
        "abm50": abm50,
        "abm100": abm100,
        "abm_all": pd.concat([d for d in [abm50, abm100] if len(d)], ignore_index=True, sort=False) if (len(abm50) or len(abm100)) else pd.DataFrame(),
    }

# -----------------------------
# Служебные функции для графиков
# -----------------------------

def ensure_out(out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def save_both(fig: plt.Figure, out_dir: Path, stem: str) -> None:
    out_dir = ensure_out(out_dir)
    png = out_dir / f"{stem}.png"
    svg = out_dir / f"{stem}.svg"
    fig.savefig(png, bbox_inches="tight", facecolor="white")
    fig.savefig(svg, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"[OK] {png}")


def placeholder(out_dir: Path, stem: str, title: str, reason: str) -> None:
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.axis("off")
    ax.text(0.5, 0.62, title, ha="center", va="center", fontsize=16, weight="bold")
    ax.text(0.5, 0.42, reason, ha="center", va="center", fontsize=11, color=PALETTE["gray"], wrap=True)
    save_both(fig, out_dir, stem)


def ordered_unique(values: Iterable[object], preferred: List[str]) -> List[str]:
    vals = [v for v in pd.Series(list(values)).dropna().astype(str).unique().tolist()]
    return [v for v in preferred if v in vals] + [v for v in vals if v not in preferred]


def draw_box(ax, xy, w, h, text, fc, ec="#334155", fontsize=10, lw=1.2):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=lw, edgecolor=ec, facecolor=fc
    )
    ax.add_patch(box)
    ax.text(xy[0] + w / 2, xy[1] + h / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)
    return box


def arrow(ax, start, end, color="#334155"):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=14, linewidth=1.3, color=color))

# -----------------------------
# Схемы методологии
# -----------------------------

def fig_methodology_pipeline(out_dir: Path):
    fig, ax = plt.subplots(figsize=(13, 5.2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title("Гибридный пайплайн оценки гендерных смещений в LLM/SLM-агентах", pad=18, weight="bold")

    boxes = [
        (0.03, 0.55, 0.14, 0.22, "Входные данные\nEN/RU резюме\nконтрфактические пары", PALETTE["light_gray"]),
        (0.22, 0.55, 0.13, 0.22, "Модели\nYandexGPT Lite\n5 Pro / 5.1", "#E8F1FF"),
        (0.40, 0.55, 0.13, 0.22, "Одиночные\nпроверки\nexpressed / encoded", "#E9F7F3"),
        (0.58, 0.55, 0.13, 0.22, "Статический\nfairness-аудит\nWEAT/SEAT + JobFair", "#FFF3D6"),
        (0.76, 0.55, 0.18, 0.22, "Динамическая\nABM-симуляция\n50/15 и 100/30", "#F2E8FF"),
    ]
    centers = []
    for x, y, w, h, text, fc in boxes:
        draw_box(ax, (x, y), w, h, text, fc, fontsize=10)
        centers.append((x + w, y + h/2, x, y + h/2))
    for i in range(len(boxes) - 1):
        arrow(ax, (boxes[i][0] + boxes[i][2] + 0.015, boxes[i][1] + boxes[i][3]/2),
              (boxes[i+1][0] - 0.015, boxes[i+1][1] + boxes[i+1][3]/2))

    # Нижний слой метрик
    metric_boxes = [
        (0.19, 0.18, 0.18, 0.18, "Уровень представлений:\nprobe accuracy, WEAT d", "#F8FAFC"),
        (0.41, 0.18, 0.18, 0.18, "Уровень генерации:\nexpressed bias, gendered response", "#F8FAFC"),
        (0.63, 0.18, 0.18, 0.18, "Уровень решений:\nscore gap, impact ratio", "#F8FAFC"),
        (0.84, 0.18, 0.13, 0.18, "Системный уровень:\nDP diff, adverse impact", "#F8FAFC"),
    ]
    for x, y, w, h, text, fc in metric_boxes:
        draw_box(ax, (x, y), w, h, text, fc, fontsize=9, lw=0.9)
    ax.text(0.03, 0.27, "Единая логика:\nот скрытых ассоциаций\nк итоговому распределению найма", fontsize=10, va="center", color=PALETTE["dark"])
    save_both(fig, out_dir, "01_methodology_pipeline")


def fig_abm_architecture(out_dir: Path):
    fig, ax = plt.subplots(figsize=(12, 6.5))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title("Архитектура ABM/LLM-системы найма", pad=18, weight="bold")

    draw_box(ax, (0.05, 0.70), 0.22, 0.16, "Mesa-среда\nкандидаты, вакансии, раунды,\nстатус найма, сбор метрик", "#E8F1FF")
    draw_box(ax, (0.39, 0.70), 0.22, 0.16, "Orchestration layer\nпоследовательность вызовов,\nвалидация, парсинг", "#E9F7F3")
    draw_box(ax, (0.73, 0.70), 0.20, 0.16, "LLM-рекрутёр\nscore, hire/reject,\nкраткое обоснование", "#FFF3D6")

    draw_box(ax, (0.08, 0.36), 0.20, 0.14, "Fairness-сценарии\nnone / demographic parity\nSoft Auditor / anonymized", "#F2E8FF", fontsize=9)
    draw_box(ax, (0.40, 0.36), 0.20, 0.14, "Soft Auditor\nпроверка решения\nи fairness-сигнала", "#FFECEC", fontsize=9)
    draw_box(ax, (0.72, 0.36), 0.20, 0.14, "Метрики\nDP diff, impact ratio,\nscore gap, Gini", "#F8FAFC", fontsize=9)

    arrow(ax, (0.28, 0.78), (0.38, 0.78))
    arrow(ax, (0.62, 0.78), (0.72, 0.78))
    arrow(ax, (0.83, 0.69), (0.83, 0.52))
    arrow(ax, (0.72, 0.43), (0.61, 0.43))
    arrow(ax, (0.39, 0.43), (0.29, 0.43))
    arrow(ax, (0.18, 0.51), (0.18, 0.69))
    arrow(ax, (0.50, 0.51), (0.50, 0.69))
    arrow(ax, (0.61, 0.43), (0.71, 0.43))

    ax.text(0.5, 0.13,
            "Смысл схемы: Mesa хранит состояние симуляции, orchestration layer управляет LLM-вызовами,\n"
            "а итоговые fairness-метрики рассчитываются по результатам многошагового процесса найма.",
            ha="center", fontsize=10, color=PALETTE["gray"])
    save_both(fig, out_dir, "02_abm_llm_architecture")


def fig_bias_levels(out_dir: Path):
    fig, ax = plt.subplots(figsize=(11.5, 5.7))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title("Уровни проявления гендерных смещений в исследовании", pad=18, weight="bold")
    levels = [
        ("Encoded / implicit", "ассоциации в представлениях\nembeddings, probing, WEAT/SEAT", "#E8F1FF"),
        ("Expressed", "гендерно маркированные\nответы и объяснения", "#E9F7F3"),
        ("Counterfactual", "различия в оценке\nпарных резюме JobFair", "#FFF3D6"),
        ("Systemic / ABM", "итоговые различия\nв найме и impact ratio", "#F2E8FF"),
    ]
    xs = [0.06, 0.30, 0.54, 0.78]
    for i, ((title, desc, fc), x) in enumerate(zip(levels, xs)):
        draw_box(ax, (x, 0.44), 0.18, 0.26, f"{title}\n\n{desc}", fc, fontsize=9.5)
        ax.text(x+0.09, 0.30, f"Уровень {i+1}", ha="center", va="center", fontsize=10, weight="bold", color=PALETTE["dark"])
        if i < 3:
            arrow(ax, (x+0.19, 0.57), (xs[i+1]-0.01, 0.57))
    ax.text(0.5, 0.13,
            "Одного уровня проверки недостаточно: внешне нейтральная генерация может сосуществовать\nсо скрытыми ассоциациями или системными различиями в динамической симуляции.",
            ha="center", fontsize=10, color=PALETTE["gray"])
    save_both(fig, out_dir, "03_bias_levels_map")

# -----------------------------
# Визуализации результатов
# -----------------------------

def fig_model_stage_matrix(data: Dict[str, pd.DataFrame], out_dir: Path):
    stages = ["expressed", "encoded", "weat", "jobfair", "abm50", "abm100"]
    rows = MODEL_ORDER
    matrix = np.zeros((len(rows), len(stages)))
    for j, st in enumerate(stages):
        df = data.get(st, pd.DataFrame())
        if len(df) and "model_display" in df.columns:
            available = set(df["model_display"].dropna())
            for i, m in enumerate(rows):
                matrix[i, j] = 1 if m in available else 0
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap="Greens")
    ax.set_xticks(range(len(stages)), ["Expressed", "Encoded", "WEAT/SEAT", "JobFair", "ABM 50/15", "ABM 100/30"], rotation=25, ha="right")
    ax.set_yticks(range(len(rows)), rows)
    ax.set_title("Покрытие экспериментальных стадий по моделям", weight="bold", pad=14)
    for i in range(len(rows)):
        for j in range(len(stages)):
            ax.text(j, i, "есть" if matrix[i, j] else "нет", ha="center", va="center", fontsize=9,
                    color="white" if matrix[i, j] else PALETTE["gray"])
    ax.grid(False)
    save_both(fig, out_dir, "04_model_stage_coverage")


def fig_expressed_heatmap(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["expressed"].copy()
    if df.empty:
        placeholder(out_dir, "05_expressed_bias_heatmap", "Expressed bias", "Файлы *expressed_bias_results.csv не найдены.")
        return
    metric = first_existing(df, ["expressed_bias", "bias_score", "mean_expressed_bias"])
    if metric is None:
        placeholder(out_dir, "05_expressed_bias_heatmap", "Expressed bias", "В CSV нет столбца expressed_bias / bias_score.")
        return
    df[metric] = to_num(df[metric])
    agg = df.groupby(["model_display", "language"], as_index=False)[metric].mean()
    models = ordered_unique(agg["model_display"], MODEL_ORDER)
    langs = ordered_unique(agg["language"], LANG_ORDER)
    pivot = agg.pivot(index="model_display", columns="language", values=metric).reindex(index=models, columns=langs)

    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    vals = pivot.values.astype(float)
    vmax = np.nanmax(np.abs(vals)) if np.isfinite(vals).any() else 1
    im = ax.imshow(vals, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(langs)), langs)
    ax.set_yticks(range(len(models)), models)
    ax.set_title("Выраженное гендерное смещение по моделям и языкам", weight="bold", pad=14)
    for i in range(len(models)):
        for j in range(len(langs)):
            v = vals[i, j]
            ax.text(j, i, "—" if np.isnan(v) else f"{v:.3f}", ha="center", va="center", fontsize=10)
    cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
    cbar.set_label(metric)
    ax.grid(False)
    save_both(fig, out_dir, "05_expressed_bias_heatmap")


def fig_gendered_response_bars(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["expressed"].copy()
    if df.empty:
        placeholder(out_dir, "06_gendered_response_pct", "Доля гендерно маркированных ответов", "Файлы expressed не найдены.")
        return
    metric = first_existing(df, ["gendered_response_pct", "gendered_pct", "non_neutral_pct"])
    if metric is None:
        placeholder(out_dir, "06_gendered_response_pct", "Доля гендерно маркированных ответов", "В CSV нет столбца gendered_response_pct.")
        return
    df[metric] = to_num(df[metric])
    agg = df.groupby(["model_display", "language"], as_index=False)[metric].mean()
    models = ordered_unique(agg["model_display"], MODEL_ORDER)
    x = np.arange(len(models))
    width = 0.35
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for idx, lang in enumerate(LANG_ORDER):
        vals = agg[agg["language"] == lang].set_index("model_display").reindex(models)[metric].values
        ax.bar(x + (idx - 0.5)*width, vals, width, label=lang)
    ax.set_xticks(x, models, rotation=15, ha="right")
    ax.set_ylabel(metric)
    ax.set_title("Гендерно маркированные ответы: EN vs RU", weight="bold", pad=14)
    ax.legend(title="Язык")
    save_both(fig, out_dir, "06_gendered_response_pct")


def fig_encoded_alignment(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["encoded"].copy()
    if df.empty:
        placeholder(out_dir, "07_encoded_alignment", "Encoded bias и alignment gap", "Файлы *encoded_results.csv не найдены.")
        return
    acc_col = first_existing(df, ["probe_accuracy", "accuracy", "probe_acc"])
    gap_col = first_existing(df, ["alignment_gap", "gap"])
    jr_col = first_existing(df, ["jailbreak_reactivation", "reactivation", "jailbreak_reactivation_score"])
    if acc_col is None and gap_col is None and jr_col is None:
        placeholder(out_dir, "07_encoded_alignment", "Encoded bias и alignment gap", "В CSV нет probe_accuracy / alignment_gap / jailbreak_reactivation.")
        return
    for col in [acc_col, gap_col, jr_col]:
        if col:
            df[col] = to_num(df[col])
    agg_cols = [c for c in [acc_col, gap_col, jr_col] if c]
    agg = df.groupby(["model_display", "language"], as_index=False)[agg_cols].mean()

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    xcol = acc_col or gap_col or jr_col
    ycol = gap_col or jr_col or acc_col
    if xcol == ycol:
        ycol = None
    if ycol:
        for lang in ordered_unique(agg["language"], LANG_ORDER):
            sub = agg[agg["language"] == lang]
            ax.scatter(sub[xcol], sub[ycol], s=120, label=lang, alpha=0.85)
            for _, r in sub.iterrows():
                ax.annotate(r["model_display"].replace("YandexGPT ", ""), (r[xcol], r[ycol]), xytext=(6, 6), textcoords="offset points", fontsize=8)
        ax.set_xlabel(xcol)
        ax.set_ylabel(ycol)
    else:
        sub = agg.groupby("model_display", as_index=False)[xcol].mean()
        ax.bar(sub["model_display"], sub[xcol])
        ax.set_ylabel(xcol)
        ax.tick_params(axis="x", rotation=15)
    ax.set_title("Encoded bias: точность probing и разрыв выравнивания", weight="bold", pad=14)
    ax.legend(title="Язык") if ycol else None
    save_both(fig, out_dir, "07_encoded_alignment")


def fig_weat_lollipop(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["weat"].copy()
    if df.empty:
        placeholder(out_dir, "08_weat_effect_sizes", "WEAT/SEAT effect sizes", "Файлы *weat_results.csv не найдены.")
        return
    effect_col = first_existing(df, ["effect_size_d", "effect_size", "cohens_d", "d"])
    p_col = first_existing(df, ["p_value", "p", "pval"])
    test_col = first_existing(df, ["test", "test_name", "weat_test"])
    if effect_col is None:
        placeholder(out_dir, "08_weat_effect_sizes", "WEAT/SEAT effect sizes", "В CSV нет столбца effect_size_d/effect_size.")
        return
    df[effect_col] = to_num(df[effect_col])
    if p_col:
        df[p_col] = to_num(df[p_col])
    df["label"] = df["model_display"] + " / " + df["language"].astype(str)
    if test_col:
        df["label"] += " / " + df[test_col].astype(str)
    plot_df = df.sort_values(effect_col).tail(30)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.30*len(plot_df)+1.5)))
    y = np.arange(len(plot_df))
    ax.hlines(y, 0, plot_df[effect_col], color="#94A3B8", linewidth=2)
    sig = plot_df[p_col] < 0.05 if p_col else pd.Series(False, index=plot_df.index)
    colors = np.where(sig, PALETTE["orange"], PALETTE["blue"])
    ax.scatter(plot_df[effect_col], y, s=70, c=colors, zorder=3)
    ax.axvline(0, color="#111827", linewidth=1)
    ax.set_yticks(y, plot_df["label"])
    ax.set_xlabel(effect_col)
    ax.set_title("WEAT/SEAT: размеры эффекта по тестам", weight="bold", pad=14)
    if p_col:
        ax.legend(handles=[Line2D([0], [0], marker='o', color='w', label='p < 0.05', markerfacecolor=PALETTE["orange"], markersize=8),
                           Line2D([0], [0], marker='o', color='w', label='p ≥ 0.05', markerfacecolor=PALETTE["blue"], markersize=8)],
                  loc="lower right")
    save_both(fig, out_dir, "08_weat_effect_sizes")


def fig_jobfair_compression(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["jobfair"].copy()
    if df.empty:
        placeholder(out_dir, "09_jobfair_score_compression", "JobFair score compression", "Файлы *jobfair_results.csv не найдены.")
        return
    score_cols = [c for c in ["mean_score_male", "mean_score_female", "mean_score_neutral"] if c in df.columns]
    if len(score_cols) < 2:
        placeholder(out_dir, "09_jobfair_score_compression", "JobFair score compression", "В CSV нет mean_score_male/female/neutral.")
        return
    for c in score_cols:
        df[c] = to_num(df[c])
    df["score_range"] = df[score_cols].max(axis=1) - df[score_cols].min(axis=1)
    group_cols = ["model_display", "language"]
    if "industry" in df.columns:
        group_cols.append("industry")
    agg = df.groupby(group_cols, as_index=False)["score_range"].mean()
    agg = agg.sort_values("score_range", ascending=False).head(25)
    fig, ax = plt.subplots(figsize=(10, max(4.8, 0.28*len(agg)+1.2)))
    labels = agg.apply(lambda r: f"{r['model_display']} / {r['language']}" + (f" / {r['industry']}" if 'industry' in agg.columns else ""), axis=1)
    ax.barh(np.arange(len(agg)), agg["score_range"])
    ax.set_yticks(np.arange(len(agg)), labels)
    ax.invert_yaxis()
    ax.set_xlabel("Размах средних score между male/female/neutral")
    ax.set_title("JobFair: насколько различаются средние оценки контрфактических резюме", weight="bold", pad=14)
    save_both(fig, out_dir, "09_jobfair_score_compression")


def fig_jobfair_raw_distributions(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["jobfair_raw"].copy()
    if df.empty:
        placeholder(out_dir, "10_jobfair_raw_score_distribution", "Raw JobFair distributions", "Raw-файлы 1_*jobfair_raw_* не найдены. График опциональный.")
        return
    score_col = first_existing(df, ["score", "parsed_score", "rating", "candidate_score"])
    gender_col = first_existing(df, ["gender", "candidate_gender", "variant", "condition"])
    if score_col is None or gender_col is None:
        placeholder(out_dir, "10_jobfair_raw_score_distribution", "Raw JobFair distributions", "В raw CSV нет score и/или gender/variant.")
        return
    df[score_col] = to_num(df[score_col])
    df = df.dropna(subset=[score_col])
    if df.empty:
        placeholder(out_dir, "10_jobfair_raw_score_distribution", "Raw JobFair distributions", "Score не распарсился в числа.")
        return
    # Boxplot по гендерным вариантам и языкам
    groups = []
    labels = []
    for lang in ordered_unique(df["language"], LANG_ORDER):
        for g in ordered_unique(df[gender_col].astype(str), ["male", "female", "neutral"]):
            vals = df[(df["language"] == lang) & (df[gender_col].astype(str) == g)][score_col].dropna().values
            if len(vals):
                groups.append(vals)
                labels.append(f"{lang}\n{g}")
    fig, ax = plt.subplots(figsize=(10, 5.2))
    ax.boxplot(groups, labels=labels, showfliers=False, patch_artist=True)
    ax.set_ylabel(score_col)
    ax.set_title("JobFair raw: распределения score по языкам и гендерным вариантам", weight="bold", pad=14)
    save_both(fig, out_dir, "10_jobfair_raw_score_distribution")


def abm_heatmap(data: Dict[str, pd.DataFrame], out_dir: Path, key: str, stem: str, title: str):
    df = data[key].copy()
    if df.empty:
        placeholder(out_dir, stem, title, f"Файлы для {key} не найдены.")
        return
    metric = first_existing(df, ["impact_ratio", "impact", "IR"])
    if metric is None:
        placeholder(out_dir, stem, title, "В CSV нет impact_ratio.")
        return
    df[metric] = to_num(df[metric])
    scen_col = first_existing(df, ["scenario", "intervention", "fairness_scenario", "condition"])
    if scen_col is None:
        df["scenario"] = "none"
        scen_col = "scenario"
    group_cols = ["model_display", "language", scen_col]
    agg = df.groupby(group_cols, as_index=False)[metric].mean()
    agg["row"] = agg["model_display"] + " / " + agg["language"]
    rows = ordered_unique(agg["row"], [m + " / " + l for m in MODEL_ORDER for l in LANG_ORDER])
    cols = ordered_unique(agg[scen_col], SCENARIO_ORDER)
    pivot = agg.pivot(index="row", columns=scen_col, values=metric).reindex(index=rows, columns=cols)
    vals = pivot.values.astype(float)
    fig, ax = plt.subplots(figsize=(10, max(4.5, 0.42*len(rows)+1.2)))
    im = ax.imshow(vals, cmap="RdYlGn", vmin=0, vmax=max(1.2, np.nanmax(vals) if np.isfinite(vals).any() else 1.2))
    ax.set_xticks(range(len(cols)), cols, rotation=25, ha="right")
    ax.set_yticks(range(len(rows)), rows)
    ax.set_title(title, weight="bold", pad=14)
    for i in range(len(rows)):
        for j in range(len(cols)):
            v = vals[i, j]
            txt = "—" if np.isnan(v) else f"{v:.2f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9)
    ax.axvline(-0.5, color="none")
    cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
    cbar.set_label("impact_ratio")
    ax.grid(False)
    save_both(fig, out_dir, stem)


def fig_abm_scale_shift(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["abm_all"].copy()
    if df.empty:
        placeholder(out_dir, "13_abm_scale_shift", "Масштабирование ABM", "ABM-файлы не найдены.")
        return
    metric = first_existing(df, ["impact_ratio", "impact", "IR"])
    if metric is None:
        placeholder(out_dir, "13_abm_scale_shift", "Масштабирование ABM", "В CSV нет impact_ratio.")
        return
    scen_col = first_existing(df, ["scenario", "intervention", "fairness_scenario", "condition"])
    if scen_col is None:
        df["scenario"] = "none"
        scen_col = "scenario"
    df[metric] = to_num(df[metric])
    # baseline/none для сопоставления масштаба
    base = df[df[scen_col].astype(str).str.lower().isin(["none", "baseline"])]
    if base.empty:
        base = df
    agg = base.groupby(["model_display", "language", "abm_scale"], as_index=False)[metric].mean()
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    for (m, lang), sub in agg.groupby(["model_display", "language"]):
        sub = sub.set_index("abm_scale").reindex(["50/15", "100/30"]).reset_index()
        if sub[metric].notna().sum() >= 1:
            ax.plot(sub["abm_scale"], sub[metric], marker="o", linewidth=2, label=f"{m} / {lang}")
    ax.axhline(0.8, color=PALETTE["red"], linestyle="--", linewidth=1.2, label="adverse impact threshold = 0.8")
    ax.set_ylabel("impact_ratio")
    ax.set_xlabel("Масштаб ABM")
    ax.set_title("Изменение impact ratio при масштабировании ABM", weight="bold", pad=14)
    ax.legend(ncol=2, frameon=True)
    save_both(fig, out_dir, "13_abm_scale_shift")


def fig_interventions(data: Dict[str, pd.DataFrame], out_dir: Path):
    df = data["abm_all"].copy()
    if df.empty:
        placeholder(out_dir, "14_intervention_comparison", "Сравнение fairness-интервенций", "ABM-файлы не найдены.")
        return
    metric = first_existing(df, ["impact_ratio", "impact", "IR"])
    scen_col = first_existing(df, ["scenario", "intervention", "fairness_scenario", "condition"])
    if metric is None or scen_col is None:
        placeholder(out_dir, "14_intervention_comparison", "Сравнение fairness-интервенций", "В CSV нет impact_ratio и/или scenario/intervention.")
        return
    df[metric] = to_num(df[metric])
    agg = df.groupby([scen_col, "abm_scale"], as_index=False)[metric].mean()
    scenarios = ordered_unique(agg[scen_col], SCENARIO_ORDER)
    x = np.arange(len(scenarios))
    width = 0.36
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    for i, scale in enumerate(["50/15", "100/30"]):
        vals = agg[agg["abm_scale"] == scale].set_index(scen_col).reindex(scenarios)[metric].values
        ax.bar(x + (i - 0.5)*width, vals, width, label=scale)
    ax.axhline(0.8, color=PALETTE["red"], linestyle="--", linewidth=1.2, label="порог 0.8")
    ax.set_xticks(x, scenarios, rotation=20, ha="right")
    ax.set_ylabel("Средний impact_ratio")
    ax.set_title("Fairness-интервенции в ABM: сравнение impact ratio", weight="bold", pad=14)
    ax.legend()
    save_both(fig, out_dir, "14_intervention_comparison")


def fig_static_vs_dynamic(data: Dict[str, pd.DataFrame], out_dir: Path):
    job = data["jobfair"].copy()
    abm = data["abm_all"].copy()
    if job.empty or abm.empty:
        placeholder(out_dir, "15_static_vs_dynamic", "Статический JobFair vs динамическая ABM", "Нужны одновременно JobFair и ABM CSV.")
        return
    score_cols = [c for c in ["mean_score_male", "mean_score_female", "mean_score_neutral"] if c in job.columns]
    ir_col = first_existing(abm, ["impact_ratio", "impact", "IR"])
    scen_col = first_existing(abm, ["scenario", "intervention", "fairness_scenario", "condition"])
    if len(score_cols) < 2 or ir_col is None:
        placeholder(out_dir, "15_static_vs_dynamic", "Статический JobFair vs динамическая ABM", "Не хватает score columns или impact_ratio.")
        return
    for c in score_cols:
        job[c] = to_num(job[c])
    job["jobfair_range"] = job[score_cols].max(axis=1) - job[score_cols].min(axis=1)
    job_agg = job.groupby(["model_display", "language"], as_index=False)["jobfair_range"].mean()
    abm[ir_col] = to_num(abm[ir_col])
    if scen_col:
        abm_base = abm[abm[scen_col].astype(str).str.lower().isin(["none", "baseline"])]
        if abm_base.empty:
            abm_base = abm
    else:
        abm_base = abm
    abm_agg = abm_base.groupby(["model_display", "language"], as_index=False)[ir_col].mean()
    merged = job_agg.merge(abm_agg, on=["model_display", "language"], how="inner")
    if merged.empty:
        placeholder(out_dir, "15_static_vs_dynamic", "Статический JobFair vs динамическая ABM", "Нет пересечения по моделям/языкам.")
        return
    fig, ax = plt.subplots(figsize=(8.5, 5.4))
    for lang in ordered_unique(merged["language"], LANG_ORDER):
        sub = merged[merged["language"] == lang]
        ax.scatter(sub["jobfair_range"], sub[ir_col], s=130, label=lang, alpha=0.85)
        for _, r in sub.iterrows():
            ax.annotate(r["model_display"].replace("YandexGPT ", ""), (r["jobfair_range"], r[ir_col]), xytext=(7, 7), textcoords="offset points", fontsize=9)
    ax.axhline(0.8, color=PALETTE["red"], linestyle="--", linewidth=1.2)
    ax.set_xlabel("JobFair: размах средних score")
    ax.set_ylabel("ABM baseline: средний impact_ratio")
    ax.set_title("Статический скоринг и динамическая симуляция могут расходиться", weight="bold", pad=14)
    ax.legend(title="Язык")
    save_both(fig, out_dir, "15_static_vs_dynamic")


def fig_hypothesis_summary(data: Dict[str, pd.DataFrame], out_dir: Path):
    # Автоматическая мягкая сводка без сильных утверждений.
    labels = [
        "H1\nкросс-языковая\nасимметрия",
        "H2\nмежмодельная\nдифференциация",
        "H3\nencoded / expressed\nдиссоциация",
        "H4\nABM системные\nэффекты",
    ]
    statuses = []

    # H1: есть ли различия EN/RU в expressed или ABM
    h1 = 0.5
    exp = data["expressed"]
    if len(exp):
        mcol = first_existing(exp, ["expressed_bias", "bias_score", "mean_expressed_bias", "gendered_response_pct"])
        if mcol:
            tmp = exp.copy(); tmp[mcol] = to_num(tmp[mcol])
            wide = tmp.groupby("language")[mcol].mean()
            if set(["EN", "RU"]).issubset(wide.index) and abs(wide["EN"] - wide["RU"]) > 1e-9:
                h1 = 1.0
    statuses.append(h1)

    # H2: variance between models
    h2 = 0.5
    if len(exp):
        mcol = first_existing(exp, ["expressed_bias", "bias_score", "mean_expressed_bias", "gendered_response_pct"])
        if mcol:
            tmp = exp.copy(); tmp[mcol] = to_num(tmp[mcol])
            vals = tmp.groupby("model_display")[mcol].mean()
            if len(vals.dropna()) > 1 and vals.std() > 1e-9:
                h2 = 1.0
    statuses.append(h2)

    # H3: encoded detected + expressed not uniform
    enc = data["encoded"]
    h3 = 0.5 if len(enc) else 0.0
    if len(enc) and len(exp):
        h3 = 1.0
    statuses.append(h3)

    # H4: ABM present and impact ratio below threshold somewhere
    abm = data["abm_all"]
    h4 = 0.5 if len(abm) else 0.0
    ir = first_existing(abm, ["impact_ratio", "impact", "IR"]) if len(abm) else None
    if ir:
        vals = to_num(abm[ir])
        if (vals < 0.8).any():
            h4 = 1.0
    statuses.append(h4)

    fig, ax = plt.subplots(figsize=(10, 4.8))
    colors = [PALETTE["green"] if s == 1 else PALETTE["yellow"] if s == 0.5 else PALETTE["gray"] for s in statuses]
    ax.bar(np.arange(len(labels)), statuses, color=colors)
    ax.set_ylim(0, 1.15)
    ax.set_xticks(np.arange(len(labels)), labels)
    ax.set_yticks([0, 0.5, 1.0], ["нет данных", "частично", "поддерживается"])
    ax.set_title("Сводная визуальная проверка гипотез по доступным CSV", weight="bold", pad=14)
    for i, s in enumerate(statuses):
        txt = "поддерживается" if s == 1 else "частично" if s == 0.5 else "нет данных"
        ax.text(i, s + 0.04, txt, ha="center", va="bottom", fontsize=9)
    save_both(fig, out_dir, "16_hypotheses_summary")


def fig_limitations_recommendations(out_dir: Path):
    fig, ax = plt.subplots(figsize=(12, 6.2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title("Из результатов к требованиям аудита LLM/SLM в найме", pad=18, weight="bold")
    left = [
        "Ограниченный набор моделей",
        "API-модели могут обновляться",
        "Синтетические резюме ≠ реальный рынок труда",
        "Чувствительность к промптам и масштабу",
        "Static scoring не равен системному эффекту",
    ]
    right = [
        "Межмодельный аудит",
        "Фиксация версии API и конфигов",
        "Калибровка на HR-данных",
        "Robustness-checks: промпты, seed, реплики",
        "Обязательная динамическая симуляция",
    ]
    draw_box(ax, (0.06, 0.73), 0.34, 0.10, "Ограничения", "#FFECEC", fontsize=12)
    draw_box(ax, (0.60, 0.73), 0.34, 0.10, "Практические следствия", "#E9F7F3", fontsize=12)
    y0 = 0.60
    for i, (l, r) in enumerate(zip(left, right)):
        y = y0 - i*0.095
        draw_box(ax, (0.05, y), 0.36, 0.06, l, "#FFF7F7", fontsize=9, lw=0.8)
        draw_box(ax, (0.59, y), 0.36, 0.06, r, "#F3FFFA", fontsize=9, lw=0.8)
        arrow(ax, (0.42, y+0.03), (0.58, y+0.03), color=PALETTE["gray"])
    save_both(fig, out_dir, "17_limitations_to_audit_recommendations")


def write_manifest(out_dir: Path):
    rows = [
        ("01_methodology_pipeline.png", "Глава 2, после раздела 2.5 или в начале 2.8", "Схема всей гибридной методологии."),
        ("02_abm_llm_architecture.png", "Глава 2, раздел 2.5", "Архитектура Mesa + orchestration layer + LLM-рекрутёр + Soft Auditor."),
        ("03_bias_levels_map.png", "Глава 1, после 1.6 или в конце 1.7", "Теоретическая схема уровней смещения."),
        ("04_model_stage_coverage.png", "Глава 3, перед 3.1", "Покрытие стадий эксперимента по моделям."),
        ("05_expressed_bias_heatmap.png", "Глава 3, раздел 3.1", "Expressed bias по моделям и языкам."),
        ("06_gendered_response_pct.png", "Глава 3, раздел 3.1", "Доля гендерно маркированных ответов."),
        ("07_encoded_alignment.png", "Глава 3, раздел 3.1", "Encoded/probing и alignment gap."),
        ("08_weat_effect_sizes.png", "Глава 3, раздел 3.2", "WEAT/SEAT effect size по моделям/языкам."),
        ("09_jobfair_score_compression.png", "Глава 3, раздел 3.3", "Насколько JobFair-средние score различаются между вариантами резюме."),
        ("10_jobfair_raw_score_distribution.png", "Приложение или раздел 3.3", "Распределения raw JobFair score; вставлять только если raw score корректно распарсился."),
        ("11_abm_impact_ratio_50_15.png", "Глава 3, раздел 3.4", "ABM 50/15: impact ratio по моделям/языкам/сценариям."),
        ("12_abm_impact_ratio_100_30.png", "Глава 3, раздел 3.5", "ABM 100/30: impact ratio по моделям/языкам/сценариям."),
        ("13_abm_scale_shift.png", "Глава 3, конец 3.5 или глава 4, раздел 4.4", "Сравнение масштабов 50/15 и 100/30."),
        ("14_intervention_comparison.png", "Глава 3, раздел 3.6 или глава 4, раздел 4.6", "Сравнение fairness-интервенций."),
        ("15_static_vs_dynamic.png", "Глава 3, раздел 3.7 или глава 4, раздел 4.5", "Статический JobFair против динамической ABM."),
        ("16_hypotheses_summary.png", "Глава 3, раздел 3.8", "Компактная визуальная сводка гипотез."),
        ("17_limitations_to_audit_recommendations.png", "Глава 4, разделы 4.9-4.10 или заключение", "Ограничения и практические следствия."),
    ]
    df = pd.DataFrame(rows, columns=["figure", "where_to_insert", "meaning"])
    df.to_csv(out_dir / "FIGURE_INSERTION_PLAN.csv", index=False, encoding="utf-8-sig")
    md = out_dir / "FIGURE_INSERTION_PLAN.md"
    with md.open("w", encoding="utf-8") as f:
        f.write("# План вставки визуализаций в ВКР\n\n")
        for fig, where, meaning in rows:
            f.write(f"- **{fig}** — {where}. {meaning}\n")
    print(f"[OK] {md}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--results-dir", type=str, default="notebook_results", help="Папка с CSV-файлами результатов")
    parser.add_argument("--out-dir", type=str, default="vkr_figures", help="Папка для сохранения графиков")
    args = parser.parse_args(args=[])

    results_dir = Path(args.results_dir)
    out_dir = ensure_out(Path(args.out_dir))
    if not results_dir.exists():
        raise FileNotFoundError(f"Папка с результатами не найдена: {results_dir.resolve()}")

    data = load_all(results_dir)
    print("\nНайденные строки по стадиям:")
    for k, df in data.items():
        print(f"  {k:12s}: {len(df)}")
    print()

    # Схемы методологии
    fig_methodology_pipeline(out_dir)
    fig_abm_architecture(out_dir)
    fig_bias_levels(out_dir)

    # Сводки и результаты
    fig_model_stage_matrix(data, out_dir)
    fig_expressed_heatmap(data, out_dir)
    fig_gendered_response_bars(data, out_dir)
    fig_encoded_alignment(data, out_dir)
    fig_weat_lollipop(data, out_dir)
    fig_jobfair_compression(data, out_dir)
    fig_jobfair_raw_distributions(data, out_dir)
    abm_heatmap(data, out_dir, "abm50", "11_abm_impact_ratio_50_15", "ABM 50 кандидатов / квота 15: impact ratio")
    abm_heatmap(data, out_dir, "abm100", "12_abm_impact_ratio_100_30", "ABM 100 кандидатов / квота 30: impact ratio")
    fig_abm_scale_shift(data, out_dir)
    fig_interventions(data, out_dir)
    fig_static_vs_dynamic(data, out_dir)
    fig_hypothesis_summary(data, out_dir)
    fig_limitations_recommendations(out_dir)
    write_manifest(out_dir)

    print("\nГотово. Вставляй PNG в Word, SVG сохраняй как векторные исходники.")


if __name__ == "__main__":
    main()



Найденные строки по стадиям:
  expressed   : 6
  encoded     : 6
  weat        : 12
  jobfair     : 18
  jobfair_raw : 648
  abm50       : 54
  abm100      : 54
  abm_all     : 108

[OK] vkr_figures/01_methodology_pipeline.png
[OK] vkr_figures/02_abm_llm_architecture.png
[OK] vkr_figures/03_bias_levels_map.png
[OK] vkr_figures/04_model_stage_coverage.png
[OK] vkr_figures/05_expressed_bias_heatmap.png
[OK] vkr_figures/06_gendered_response_pct.png
[OK] vkr_figures/07_encoded_alignment.png
[OK] vkr_figures/08_weat_effect_sizes.png
[OK] vkr_figures/09_jobfair_score_compression.png
[OK] vkr_figures/10_jobfair_raw_score_distribution.png


/var/folders/77/38d2g36s7yg21grdkhyw9hvw0000gn/T/ipykernel_41434/3932238196.py:494: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(groups, labels=labels, showfliers=False, patch_artist=True)


[OK] vkr_figures/11_abm_impact_ratio_50_15.png
[OK] vkr_figures/12_abm_impact_ratio_100_30.png
[OK] vkr_figures/13_abm_scale_shift.png
[OK] vkr_figures/14_intervention_comparison.png
[OK] vkr_figures/15_static_vs_dynamic.png
[OK] vkr_figures/16_hypotheses_summary.png
[OK] vkr_figures/17_limitations_to_audit_recommendations.png
[OK] vkr_figures/FIGURE_INSERTION_PLAN.md

Готово. Вставляй PNG в Word, SVG сохраняй как векторные исходники.


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import ast
import json
import re

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# НАСТРОЙКИ
# =========================
RESULTS_DIR = Path.cwd()/ "notebook_results"    # <- поменяй на свою папку
OUT_DIR = Path("vkr_figures_final") # <- сюда сохранятся PNG
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_SCALE = 3
SAVE_WIDTH = 1800
SAVE_HEIGHT = 1000

# Если не нужны схемы через graphviz, поставь False
MAKE_SCHEMES = True

# =========================
# СТИЛЬ
# =========================
PLOTLY_TEMPLATE = "plotly_white"

COLORS = {
    "EN": "#4C78A8",
    "RU": "#F58518",
    "male": "#4C78A8",
    "female": "#E45756",
    "neutral": "#72B7B2",
    "none": "#9D755D",
    "baseline": "#9D755D",
    "demographic_parity": "#54A24B",
    "auditor": "#B279A2",
    "anonymized": "#FF9DA6",
    "yes": "#E45756",
    "no": "#72B7B2",
    "missing": "#BDBDBD",
}

MODEL_NAME_MAP = {
    "yandexgpt_api": "YandexGPT Lite",
    "yandexgpt-lite": "YandexGPT Lite",
    "lite": "YandexGPT Lite",

    "yandexgpt-5-pro": "YandexGPT 5 Pro",
    "yandexgpt_5_pro": "YandexGPT 5 Pro",
    "5 pro": "YandexGPT 5 Pro",

    "yandexgpt_5_1": "YandexGPT 5.1",
    "yandexgpt-5.1": "YandexGPT 5.1",
    "yandexgpt_5.1": "YandexGPT 5.1",
    "5.1": "YandexGPT 5.1",
}

MODEL_ORDER = ["YandexGPT Lite", "YandexGPT 5 Pro", "YandexGPT 5.1"]
LANG_ORDER = ["EN", "RU"]
SCENARIO_ORDER = ["none", "demographic_parity", "auditor", "anonymized"]


# =========================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================
def clean_model_name(x):
    if pd.isna(x):
        return x
    s = str(x).strip().lower()
    return MODEL_NAME_MAP.get(s, str(x))

def clean_lang(x):
    if pd.isna(x):
        return x
    s = str(x).strip().upper()
    if s in ["EN", "RU"]:
        return s
    return s

def save_png(fig, filename, width=SAVE_WIDTH, height=SAVE_HEIGHT, scale=SAVE_SCALE):
    fig.update_layout(template=PLOTLY_TEMPLATE)
    fig.write_image(str(OUT_DIR / filename), width=width, height=height, scale=scale)
    print("saved:", OUT_DIR / filename)

def find_col(df, candidates, required=False):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Не найден ни один из столбцов: {candidates}")
    return None

def parse_maybe_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return {}
        try:
            return ast.literal_eval(s)
        except Exception:
            try:
                return json.loads(s)
            except Exception:
                return {}
    return {}

def find_files(patterns):
    files = []
    for p in patterns:
        files.extend(list(RESULTS_DIR.glob(p)))
    return sorted(files)

def first_existing(patterns):
    files = find_files(patterns)
    return files[0] if files else None

def concat_if_any(dfs):
    dfs = [d for d in dfs if d is not None and len(d) > 0]
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

def scenario_normalize(x):
    if pd.isna(x):
        return x
    s = str(x).strip().lower()
    mapping = {
        "baseline": "none",
        "none": "none",
        "demographic_parity": "demographic_parity",
        "dp": "demographic_parity",
        "auditor": "auditor",
        "soft auditor": "auditor",
        "soft_auditor": "auditor",
        "anonymized": "anonymized",
        "anonymous": "anonymized"
    }
    return mapping.get(s, s)

def nice_yes_no(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"true", "1", "yes", "y"}:
            return 1
        if s in {"false", "0", "no", "n"}:
            return 0
    return int(bool(x))

def ensure_categorical_order(df, col, order):
    if col in df.columns:
        df[col] = pd.Categorical(df[col], categories=order, ordered=True)
    return df


# =========================
# ЗАГРУЗКА CSV
# =========================
def load_expressed():
    files = find_files(["*expressed_bias_results.csv"])
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        model_col = find_col(df, ["model", "model_key", "provider_model"], required=True)
        lang_col = find_col(df, ["language", "lang"], required=True)

        rename_map = {
            model_col: "model",
            lang_col: "language",
        }
        for old, new in [
            (find_col(df, ["expressed_bias"]), "expressed_bias"),
            (find_col(df, ["mean_gender_ratio"]), "mean_gender_ratio"),
            (find_col(df, ["male_dominant_pct"]), "male_dominant_pct"),
            (find_col(df, ["female_dominant_pct"]), "female_dominant_pct"),
            (find_col(df, ["neutral_pct"]), "neutral_pct"),
            (find_col(df, ["gendered_response_pct"]), "gendered_response_pct"),
        ]:
            if old is not None:
                rename_map[old] = new

        df = df.rename(columns=rename_map)
        df["model"] = df["model"].map(clean_model_name)
        df["language"] = df["language"].map(clean_lang)
        dfs.append(df)

    out = concat_if_any(dfs)
    if len(out):
        out = ensure_categorical_order(out, "model", MODEL_ORDER)
        out = ensure_categorical_order(out, "language", LANG_ORDER)
    return out

def load_encoded():
    files = find_files(["*encoded_results.csv"])
    dfs = []
    for f in files:
        df = pd.read_csv(f)

        model_col = find_col(df, ["model", "model_key", "provider_model"], required=True)
        lang_col = find_col(df, ["language", "lang"], required=True)

        rename_map = {
            model_col: "model",
            lang_col: "language",
        }
        for old, new in [
            (find_col(df, ["probe_accuracy"]), "probe_accuracy"),
            (find_col(df, ["encoded_detected"]), "encoded_detected"),
            (find_col(df, ["alignment_gap"]), "alignment_gap"),
            (find_col(df, ["jailbreak_reactivation"]), "jailbreak_reactivation"),
        ]:
            if old is not None:
                rename_map[old] = new

        df = df.rename(columns=rename_map)
        df["model"] = df["model"].map(clean_model_name)
        df["language"] = df["language"].map(clean_lang)

        if "encoded_detected" in df.columns:
            df["encoded_detected"] = df["encoded_detected"].apply(nice_yes_no)

        dfs.append(df)

    out = concat_if_any(dfs)
    if len(out):
        out = ensure_categorical_order(out, "model", MODEL_ORDER)
        out = ensure_categorical_order(out, "language", LANG_ORDER)
    return out

def load_weat():
    files = find_files(["*weat_results.csv"])
    dfs = []
    for f in files:
        df = pd.read_csv(f)

        model_col = find_col(df, ["model", "model_key", "provider_model"], required=True)
        lang_col = find_col(df, ["language", "lang"], required=True)

        rename_map = {
            model_col: "model",
            lang_col: "language",
        }
        for old, new in [
            (find_col(df, ["test", "test_name"]), "test"),
            (find_col(df, ["effect_size_d", "effect_size", "d"]), "effect_size_d"),
            (find_col(df, ["p_value", "pvalue", "p"]), "p_value"),
            (find_col(df, ["interpretation"]), "interpretation"),
            (find_col(df, ["clustering_gap"]), "clustering_gap"),
        ]:
            if old is not None:
                rename_map[old] = new

        df = df.rename(columns=rename_map)
        df["model"] = df["model"].map(clean_model_name)
        df["language"] = df["language"].map(clean_lang)
        dfs.append(df)

    out = concat_if_any(dfs)
    if len(out):
        out = ensure_categorical_order(out, "model", MODEL_ORDER)
        out = ensure_categorical_order(out, "language", LANG_ORDER)
    return out

def load_jobfair():
    files = find_files(["*jobfair_results.csv"])
    dfs = []
    for f in files:
        df = pd.read_csv(f)

        model_col = find_col(df, ["model", "model_key", "provider_model"], required=True)
        lang_col = find_col(df, ["language", "lang"], required=True)

        rename_map = {
            model_col: "model",
            lang_col: "language",
        }
        for old, new in [
            (find_col(df, ["industry", "domain"]), "industry"),
            (find_col(df, ["mean_score_male"]), "mean_score_male"),
            (find_col(df, ["mean_score_female"]), "mean_score_female"),
            (find_col(df, ["mean_score_neutral"]), "mean_score_neutral"),
            (find_col(df, ["impact_ratio"]), "impact_ratio"),
            (find_col(df, ["taste_based"]), "taste_based"),
            (find_col(df, ["bayes_factor", "BF10"]), "BF10"),
            (find_col(df, ["level_bias"]), "level_bias"),
            (find_col(df, ["spread_bias"]), "spread_bias"),
        ]:
            if old is not None:
                rename_map[old] = new

        df = df.rename(columns=rename_map)
        df["model"] = df["model"].map(clean_model_name)
        df["language"] = df["language"].map(clean_lang)

        # Парсим словари с метриками
        for col in ["level_bias", "spread_bias"]:
            if col in df.columns:
                parsed = df[col].apply(parse_maybe_dict)
                df[f"{col}_observed_diff"] = parsed.apply(lambda x: x.get("observed_diff"))
                df[f"{col}_p_value"] = parsed.apply(lambda x: x.get("p_value"))
                df[f"{col}_significant"] = parsed.apply(lambda x: x.get("significant"))
                df[f"{col}_n_permutations"] = parsed.apply(lambda x: x.get("n_permutations"))

        dfs.append(df)

    out = concat_if_any(dfs)
    if len(out):
        out = ensure_categorical_order(out, "model", MODEL_ORDER)
        out = ensure_categorical_order(out, "language", LANG_ORDER)
    return out

def load_jobfair_raw():
    files = find_files(["1_*jobfair_raw*results.csv", "*jobfair_raw*results.csv"])
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # В raw-файлах часто модель/язык уже зашиты в имени файла
        name = f.name.lower()

        model = None
        if "api" in name:
            model = "YandexGPT Lite"
        elif "5_pro" in name or "5-pro" in name:
            model = "YandexGPT 5 Pro"
        elif "5_1" in name or "5.1" in name:
            model = "YandexGPT 5.1"

        lang = "EN" if "_en_" in name or name.endswith("_en_results.csv") else "RU" if "_ru_" in name or name.endswith("_ru_results.csv") else None

        if "model" not in df.columns:
            df["model"] = model
        else:
            df["model"] = df["model"].fillna(model).map(clean_model_name)

        if "language" not in df.columns:
            df["language"] = lang
        else:
            df["language"] = df["language"].fillna(lang).map(clean_lang)

        # Ищем стандартные поля
        score_col = find_col(df, ["score", "final_score"], required=False)
        variant_col = find_col(df, ["variant", "gender_variant", "candidate_variant", "gender"], required=False)
        industry_col = find_col(df, ["industry", "domain"], required=False)

        rename_map = {}
        if score_col: rename_map[score_col] = "score"
        if variant_col: rename_map[variant_col] = "variant"
        if industry_col: rename_map[industry_col] = "industry"
        df = df.rename(columns=rename_map)

        dfs.append(df)

    out = concat_if_any(dfs)
    if len(out):
        out["model"] = out["model"].map(clean_model_name)
        out["language"] = out["language"].map(clean_lang)
        if "variant" in out.columns:
            out["variant"] = out["variant"].astype(str).str.lower()
    return out

def load_abm():
    files = find_files(["*abm_n50_hq15_results.csv", "*abm_n100_hq30_results.csv"])
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        name = f.name.lower()

        if "n50" in name:
            scenario_scale = "50/15"
            ncfg = 50
            hq = 15
        elif "n100" in name:
            scenario_scale = "100/30"
            ncfg = 100
            hq = 30
        else:
            scenario_scale = None
            ncfg = np.nan
            hq = np.nan

        model_col = find_col(df, ["model", "model_key", "provider_model"], required=True)
        lang_col = find_col(df, ["language", "lang"], required=True)

        rename_map = {
            model_col: "model",
            lang_col: "language",
        }

        for old, new in [
            (find_col(df, ["industry", "domain"]), "industry"),
            (find_col(df, ["scenario", "intervention"]), "scenario"),
            (find_col(df, ["male_hire_rate"]), "male_hire_rate"),
            (find_col(df, ["female_hire_rate"]), "female_hire_rate"),
            (find_col(df, ["demographic_parity_diff", "dp_diff"]), "demographic_parity_diff"),
            (find_col(df, ["score_gap"]), "score_gap"),
            (find_col(df, ["impact_ratio"]), "impact_ratio"),
            (find_col(df, ["adverse_impact"]), "adverse_impact"),
            (find_col(df, ["gini_scores"]), "gini_scores"),
            (find_col(df, ["n_candidates_config"]), "n_candidates_config"),
            (find_col(df, ["hire_quota"]), "hire_quota"),
        ]:
            if old is not None:
                rename_map[old] = new

        df = df.rename(columns=rename_map)
        df["model"] = df["model"].map(clean_model_name)
        df["language"] = df["language"].map(clean_lang)
        df["scenario"] = df["scenario"].apply(scenario_normalize)
        df["abm_scale"] = scenario_scale

        if "n_candidates_config" not in df.columns:
            df["n_candidates_config"] = ncfg
        if "hire_quota" not in df.columns:
            df["hire_quota"] = hq

        if "adverse_impact" in df.columns:
            def to_ai(v):
                if pd.isna(v):
                    return np.nan
                if isinstance(v, str):
                    s = v.strip().lower()
                    if s in {"true", "1", "yes"}:
                        return 1
                    if s in {"false", "0", "no"}:
                        return 0
                return int(bool(v))
            df["adverse_impact"] = df["adverse_impact"].apply(to_ai)

        dfs.append(df)

    out = concat_if_any(dfs)
    if len(out):
        out = ensure_categorical_order(out, "model", MODEL_ORDER)
        out = ensure_categorical_order(out, "language", LANG_ORDER)
    return out


# =========================
# ЗАГРУЗКА ДАННЫХ
# =========================
expressed = load_expressed()
encoded = load_encoded()
weat = load_weat()
jobfair = load_jobfair()
jobfair_raw = load_jobfair_raw()
abm = load_abm()

print("expressed:", expressed.shape)
print("encoded:", encoded.shape)
print("weat:", weat.shape)
print("jobfair:", jobfair.shape)
print("jobfair_raw:", jobfair_raw.shape)
print("abm:", abm.shape)


# =========================
# ФИГУРА 1
# Encoded bias — heatmap вместо сбитого scatter
# =========================
def fig01_encoded_heatmap(df):
    if df.empty:
        return

    cols_needed = ["model", "language", "probe_accuracy", "alignment_gap", "jailbreak_reactivation"]
    available = [c for c in cols_needed if c in df.columns]
    d = df[available].copy()

    d["row"] = d["model"].astype(str) + " / " + d["language"].astype(str)

    melt_cols = [c for c in ["probe_accuracy", "alignment_gap", "jailbreak_reactivation"] if c in d.columns]
    dm = d.melt(id_vars=["row"], value_vars=melt_cols, var_name="metric", value_name="value")

    fig = px.imshow(
        dm.pivot(index="row", columns="metric", values="value"),
        text_auto=".2f",
        color_continuous_scale="Blues",
        aspect="auto",
        title="Encoded bias: probing, разрыв выравнивания и реактивация",
    )
    fig.update_layout(
        font=dict(size=18),
        title_font=dict(size=30),
        coloraxis_colorbar_title="Значение",
        margin=dict(l=40, r=40, t=90, b=40),
    )
    fig.update_xaxes(title="")
    fig.update_yaxes(title="")
    save_png(fig, "fig_01_encoded_heatmap.png", height=900)

fig01_encoded_heatmap(encoded)


# =========================
# ФИГУРА 2
# WEAT/SEAT — более чистый lollipop
# =========================
def fig02_weat_lollipop(df):
    if df.empty or "effect_size_d" not in df.columns:
        return

    d = df.copy()
    if "p_value" in d.columns:
        d["sig"] = np.where(d["p_value"] < 0.05, "p < 0.05", "p ≥ 0.05")
    else:
        d["sig"] = "p неизвестно"

    d["label"] = d["model"].astype(str) + " / " + d["language"].astype(str) + " / " + d["test"].astype(str)
    d = d.sort_values("effect_size_d")

    fig = go.Figure()

    for _, row in d.iterrows():
        fig.add_trace(go.Scatter(
            x=[0, row["effect_size_d"]],
            y=[row["label"], row["label"]],
            mode="lines",
            line=dict(color="#A0AEC0", width=4),
            showlegend=False,
            hoverinfo="skip"
        ))

    color_map = {"p < 0.05": "#E45756", "p ≥ 0.05": "#4C78A8", "p неизвестно": "#9E9E9E"}

    for sig_value in d["sig"].unique():
        dd = d[d["sig"] == sig_value]
        fig.add_trace(go.Scatter(
            x=dd["effect_size_d"],
            y=dd["label"],
            mode="markers",
            marker=dict(size=16, color=color_map.get(sig_value, "#4C78A8")),
            name=sig_value,
            text=[f"d={x:.3f}" if pd.notna(x) else "" for x in dd["effect_size_d"]],
            hovertemplate="%{y}<br>effect size d=%{x:.3f}<extra></extra>"
        ))

    fig.add_vline(x=0, line_width=2, line_dash="solid", line_color="black")
    fig.update_layout(
        title="WEAT/SEAT: размеры эффекта по тестам",
        font=dict(size=18),
        title_font=dict(size=30),
        xaxis_title="effect_size_d",
        yaxis_title="",
        legend_title="Значимость",
        margin=dict(l=40, r=40, t=90, b=40),
        height=max(800, 60 * len(d) + 250)
    )
    save_png(fig, "fig_02_weat_lollipop.png", height=max(800, 60 * len(d) + 250))

fig02_weat_lollipop(weat)


# =========================
# ФИГУРА 3
# JobFair — dumbbell по male/female/neutral mean score
# =========================
def fig03_jobfair_dumbbell(df):
    if df.empty:
        return
    need = ["model", "language", "industry", "mean_score_male", "mean_score_female", "mean_score_neutral"]
    if not all(c in df.columns for c in need):
        return

    d = df[need].copy()
    d["row"] = d["model"].astype(str) + " / " + d["language"].astype(str) + " / " + d["industry"].astype(str)

    d = d.sort_values(["model", "language", "industry"]).reset_index(drop=True)

    fig = go.Figure()

    for _, row in d.iterrows():
        values = [row["mean_score_male"], row["mean_score_female"], row["mean_score_neutral"]]
        minv, maxv = np.nanmin(values), np.nanmax(values)

        fig.add_trace(go.Scatter(
            x=[minv, maxv],
            y=[row["row"], row["row"]],
            mode="lines",
            line=dict(color="#C7CBD1", width=4),
            showlegend=False,
            hoverinfo="skip"
        ))

    for variant, color, col in [
        ("male", COLORS["male"], "mean_score_male"),
        ("female", COLORS["female"], "mean_score_female"),
        ("neutral", COLORS["neutral"], "mean_score_neutral"),
    ]:
        fig.add_trace(go.Scatter(
            x=d[col],
            y=d["row"],
            mode="markers",
            marker=dict(size=14, color=color, line=dict(color="white", width=1)),
            name=variant,
            hovertemplate="%{y}<br>" + variant + ": %{x:.2f}<extra></extra>"
        ))

    fig.update_layout(
        title="JobFair: средние оценки контрфактических резюме",
        font=dict(size=18),
        title_font=dict(size=30),
        xaxis_title="Средний score",
        yaxis_title="",
        legend_title="Вариант",
        margin=dict(l=40, r=40, t=90, b=40),
        height=max(850, 55 * len(d) + 250),
    )
    save_png(fig, "fig_03_jobfair_dumbbell.png", height=max(850, 55 * len(d) + 250))

fig03_jobfair_dumbbell(jobfair)


# =========================
# ФИГУРА 5
# Expressed bias — 100% stacked bars
# =========================
def fig05_expressed_stacked(df):
    if df.empty:
        return
    needed = ["model", "language", "male_dominant_pct", "female_dominant_pct", "neutral_pct"]
    if not all(c in df.columns for c in needed):
        return

    d = df[needed].copy()
    d["row"] = d["model"].astype(str) + " / " + d["language"].astype(str)

    fig = go.Figure()
    for label, color, col in [
        ("male-dominant", COLORS["male"], "male_dominant_pct"),
        ("female-dominant", COLORS["female"], "female_dominant_pct"),
        ("neutral", COLORS["neutral"], "neutral_pct"),
    ]:
        fig.add_trace(go.Bar(
            y=d["row"],
            x=d[col],
            orientation="h",
            name=label,
            marker_color=color,
            text=[f"{v:.1f}%" if pd.notna(v) else "" for v in d[col]],
            textposition="inside"
        ))

    fig.update_layout(
        barmode="stack",
        title="Expressed bias: структура ответов по типу доминирования",
        font=dict(size=18),
        title_font=dict(size=28),
        xaxis_title="Доля ответов, %",
        yaxis_title="",
        legend_title="Тип ответа",
        margin=dict(l=40, r=40, t=90, b=40),
        height=800
    )
    save_png(fig, "fig_05_expressed_stacked.png", height=800)

fig05_expressed_stacked(expressed)


# =========================
# ФИГУРА 6
# Expressed bias — diverging bars
# =========================
def fig06_expressed_diverging(df):
    if df.empty or "expressed_bias" not in df.columns:
        return

    d = df[["model", "language", "expressed_bias"]].copy()
    d["row"] = d["model"].astype(str) + " / " + d["language"].astype(str)
    d = d.sort_values("expressed_bias")

    fig = go.Figure(go.Bar(
        y=d["row"],
        x=d["expressed_bias"],
        orientation="h",
        marker=dict(
            color=np.where(d["expressed_bias"] >= 0, COLORS["male"], COLORS["female"])
        ),
        text=[f"{v:.3f}" for v in d["expressed_bias"]],
        textposition="outside",
        hovertemplate="%{y}<br>expressed_bias=%{x:.3f}<extra></extra>"
    ))

    fig.add_vline(x=0, line_width=2, line_color="black")
    fig.update_layout(
        title="Expressed bias: направление и величина смещения",
        font=dict(size=18),
        title_font=dict(size=28),
        xaxis_title="expressed_bias",
        yaxis_title="",
        margin=dict(l=40, r=60, t=90, b=40),
        height=800
    )
    save_png(fig, "fig_06_expressed_diverging.png", height=800)

fig06_expressed_diverging(expressed)


# =========================
# ФИГУРА 7
# ABM baseline — dumbbell male vs female hire rate
# =========================
def fig07_abm_baseline_dumbbell(df):
    if df.empty:
        return
    need = ["model", "language", "industry", "abm_scale", "scenario", "male_hire_rate", "female_hire_rate"]
    if not all(c in df.columns for c in need):
        return

    d = df[df["scenario"] == "none"].copy()
    if d.empty:
        return

    d["row"] = d["model"].astype(str) + " / " + d["language"].astype(str) + " / " + d["industry"].astype(str) + " / " + d["abm_scale"].astype(str)
    d = d.sort_values(["abm_scale", "model", "language", "industry"])

    fig = go.Figure()

    for _, row in d.iterrows():
        fig.add_trace(go.Scatter(
            x=[row["male_hire_rate"], row["female_hire_rate"]],
            y=[row["row"], row["row"]],
            mode="lines",
            line=dict(color="#C7CBD1", width=4),
            showlegend=False,
            hoverinfo="skip"
        ))

    fig.add_trace(go.Scatter(
        x=d["male_hire_rate"],
        y=d["row"],
        mode="markers",
        marker=dict(size=14, color=COLORS["male"]),
        name="male",
        hovertemplate="%{y}<br>male_hire_rate=%{x:.3f}<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=d["female_hire_rate"],
        y=d["row"],
        mode="markers",
        marker=dict(size=14, color=COLORS["female"]),
        name="female",
        hovertemplate="%{y}<br>female_hire_rate=%{x:.3f}<extra></extra>"
    ))

    fig.update_layout(
        title="ABM baseline: сравнение male и female hire rate",
        font=dict(size=18),
        title_font=dict(size=28),
        xaxis_title="Hire rate",
        yaxis_title="",
        legend_title="Группа",
        margin=dict(l=40, r=40, t=90, b=40),
        height=max(900, 55 * len(d) + 250)
    )
    save_png(fig, "fig_07_abm_baseline_dumbbell.png", height=max(900, 55 * len(d) + 250))

fig07_abm_baseline_dumbbell(abm)


# =========================
# ФИГУРА 8
# Сравнение масштаба 50/15 vs 100/30 — heatmap по dp diff
# =========================
def fig08_abm_scale_heatmap(df):
    if df.empty or "demographic_parity_diff" not in df.columns:
        return

    d = df[df["scenario"] == "none"].copy()
    if d.empty:
        return

    d["row"] = d["model"].astype(str) + " / " + d["language"].astype(str) + " / " + d["industry"].astype(str)
    pivot = d.pivot_table(index="row", columns="abm_scale", values="demographic_parity_diff", aggfunc="mean")

    if pivot.empty:
        return

    fig = px.imshow(
        pivot,
        text_auto=".3f",
        color_continuous_scale="OrRd",
        aspect="auto",
        title="ABM baseline: как меняется demographic parity diff при масштабировании"
    )
    fig.update_layout(
        font=dict(size=18),
        title_font=dict(size=28),
        coloraxis_colorbar_title="DP diff",
        margin=dict(l=40, r=40, t=90, b=40),
        height=max(850, 45 * len(pivot) + 250)
    )
    fig.update_xaxes(title="Масштаб ABM")
    fig.update_yaxes(title="")
    save_png(fig, "fig_08_abm_scale_heatmap.png", height=max(850, 45 * len(pivot) + 250))

fig08_abm_scale_heatmap(abm)


# =========================
# ФИГУРА 9
# Покрытие и значение impact ratio по интервенциям
# =========================
def fig09_intervention_coverage(df):
    if df.empty or "impact_ratio" not in df.columns:
        return

    # полная сетка, чтобы не терять покрытия
    models = [m for m in MODEL_ORDER if m in df["model"].astype(str).unique()]
    langs = [l for l in LANG_ORDER if l in df["language"].astype(str).unique()]
    scales = sorted(df["abm_scale"].dropna().unique().tolist())
    scenarios = SCENARIO_ORDER

    full = pd.MultiIndex.from_product(
        [models, langs, scales, scenarios],
        names=["model", "language", "abm_scale", "scenario"]
    ).to_frame(index=False)

    g = df.groupby(["model", "language", "abm_scale", "scenario"], as_index=False)["impact_ratio"].mean()

    merged = full.merge(g, on=["model", "language", "abm_scale", "scenario"], how="left")
    merged["row"] = merged["model"].astype(str) + " / " + merged["language"].astype(str) + " / " + merged["abm_scale"].astype(str)

    pivot = merged.pivot_table(index="row", columns="scenario", values="impact_ratio", aggfunc="mean")

    fig = px.imshow(
        pivot,
        text_auto=".2f",
        color_continuous_scale="RdYlGn",
        zmin=0.6, zmax=1.0,
        aspect="auto",
        title="ABM: impact ratio по fairness-сценариям (с полным покрытием)"
    )

    # Показываем missing как текст
    missing_text = pivot.copy().astype(object)
    for r in missing_text.index:
        for c in missing_text.columns:
            if pd.isna(pivot.loc[r, c]):
                missing_text.loc[r, c] = "нет\nданных"
            else:
                missing_text.loc[r, c] = f"{pivot.loc[r, c]:.2f}"

    fig.data[0].text = missing_text.values
    fig.data[0].texttemplate = "%{text}"

    fig.update_layout(
        font=dict(size=18),
        title_font=dict(size=28),
        coloraxis_colorbar_title="Impact ratio",
        margin=dict(l=40, r=40, t=90, b=40),
        height=max(850, 45 * len(pivot) + 250)
    )
    fig.update_xaxes(title="Интервенция")
    fig.update_yaxes(title="")
    save_png(fig, "fig_09_intervention_coverage_heatmap.png", height=max(850, 45 * len(pivot) + 250))

fig09_intervention_coverage(abm)


# =========================
# ФИГУРА 11
# Доля adverse impact по сценариям
# =========================
def fig11_adverse_share(df):
    if df.empty or "adverse_impact" not in df.columns:
        return

    d = df.groupby(["abm_scale", "scenario"], as_index=False)["adverse_impact"].mean()
    d["share_pct"] = d["adverse_impact"] * 100

    fig = px.bar(
        d,
        x="scenario",
        y="share_pct",
        color="abm_scale",
        barmode="group",
        text="share_pct",
        title="ABM: доля конфигураций с adverse impact"
    )
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.update_layout(
        font=dict(size=18),
        title_font=dict(size=28),
        xaxis_title="Сценарий",
        yaxis_title="Доля конфигураций, %",
        legend_title="Масштаб",
        margin=dict(l=40, r=40, t=90, b=40),
        height=800
    )
    save_png(fig, "fig_11_adverse_impact_share.png", height=800)

fig11_adverse_share(abm)


# =========================
# ФИГУРА 12
# baseline -> demographic_parity
# =========================
def fig12_dp_slope(df):
    if df.empty or "demographic_parity_diff" not in df.columns:
        return

    d = df[df["scenario"].isin(["none", "demographic_parity"])].copy()
    if d.empty:
        return

    need = ["model", "language", "industry", "abm_scale", "scenario", "demographic_parity_diff"]
    d = d[need].copy()

    pivot = d.pivot_table(
        index=["model", "language", "industry", "abm_scale"],
        columns="scenario",
        values="demographic_parity_diff",
        aggfunc="mean"
    ).reset_index()

    if "none" not in pivot.columns or "demographic_parity" not in pivot.columns:
        return

    pivot["row"] = (
        pivot["model"].astype(str) + " / " +
        pivot["language"].astype(str) + " / " +
        pivot["industry"].astype(str) + " / " +
        pivot["abm_scale"].astype(str)
    )

    fig = go.Figure()

    for _, row in pivot.iterrows():
        fig.add_trace(go.Scatter(
            x=["none", "demographic_parity"],
            y=[row["none"], row["demographic_parity"]],
            mode="lines+markers",
            line=dict(color="#A0AEC0", width=2),
            marker=dict(size=10, color=["#9D755D", "#54A24B"]),
            showlegend=False,
            text=[row["row"], row["row"]],
            hovertemplate=row["row"] + "<br>%{x}: %{y:.3f}<extra></extra>"
        ))

    fig.update_layout(
        title="ABM: как demographic parity меняет demographic parity diff",
        font=dict(size=18),
        title_font=dict(size=28),
        xaxis_title="Сценарий",
        yaxis_title="Demographic parity diff",
        margin=dict(l=40, r=40, t=90, b=40),
        height=900
    )
    save_png(fig, "fig_12_demographic_parity_slope.png", height=900)

fig12_dp_slope(abm)


# =========================
# ФИГУРА 13
# delta относительно baseline — heatmap, а не тяжёлый график
# =========================
def fig13_auditor_delta_heatmap(df):
    if df.empty or "demographic_parity_diff" not in df.columns:
        return

    d = df[df["scenario"].isin(["none", "auditor"])].copy()
    if d.empty:
        return

    pivot = d.pivot_table(
        index=["model", "language", "industry", "abm_scale"],
        columns="scenario",
        values="demographic_parity_diff",
        aggfunc="mean"
    ).reset_index()

    if "none" not in pivot.columns or "auditor" not in pivot.columns:
        return

    pivot["delta_auditor_vs_none"] = pivot["auditor"] - pivot["none"]
    pivot["row"] = (
        pivot["model"].astype(str) + " / " +
        pivot["language"].astype(str) + " / " +
        pivot["industry"].astype(str) + " / " +
        pivot["abm_scale"].astype(str)
    )

    m = pivot[["row", "delta_auditor_vs_none"]].set_index("row")
    fig = px.imshow(
        m,
        text_auto=".3f",
        color_continuous_scale="RdBu_r",
        zmid=0,
        aspect="auto",
        title="ABM: изменение demographic parity diff при включении Soft Auditor"
    )
    fig.update_layout(
        font=dict(size=18),
        title_font=dict(size=28),
        coloraxis_colorbar_title="Δ auditor - none",
        margin=dict(l=40, r=40, t=90, b=40),
        height=max(850, 45 * len(m) + 250)
    )
    fig.update_xaxes(title="")
    fig.update_yaxes(title="")
    save_png(fig, "fig_13_auditor_delta_heatmap.png", height=max(850, 45 * len(m) + 250))

fig13_auditor_delta_heatmap(abm)


# =========================
# ФИГУРА 14
# Чувствительность к масштабу — impact ratio baseline
# =========================
def fig14_scale_sensitivity(df):
    if df.empty or "impact_ratio" not in df.columns:
        return

    d = df[df["scenario"] == "none"].copy()
    if d.empty:
        return

    pivot = d.pivot_table(
        index=["model", "language", "industry"],
        columns="abm_scale",
        values="impact_ratio",
        aggfunc="mean"
    ).reset_index()

    if "50/15" not in pivot.columns or "100/30" not in pivot.columns:
        return

    pivot["row"] = pivot["model"].astype(str) + " / " + pivot["language"].astype(str) + " / " + pivot["industry"].astype(str)

    fig = go.Figure()

    for _, row in pivot.iterrows():
        fig.add_trace(go.Scatter(
            x=["50/15", "100/30"],
            y=[row["50/15"], row["100/30"]],
            mode="lines+markers",
            line=dict(color="#A0AEC0", width=2),
            marker=dict(size=10, color=["#4C78A8", "#F58518"]),
            showlegend=False,
            hovertemplate=row["row"] + "<br>%{x}: %{y:.3f}<extra></extra>"
        ))

    fig.add_hline(y=0.8, line_dash="dash", line_color="#E45756", line_width=2)
    fig.update_layout(
        title="ABM baseline: чувствительность impact ratio к масштабу симуляции",
        font=dict(size=18),
        title_font=dict(size=28),
        xaxis_title="Масштаб ABM",
        yaxis_title="Impact ratio",
        margin=dict(l=40, r=40, t=90, b=40),
        height=900
    )
    save_png(fig, "fig_14_scale_sensitivity.png", height=900)

fig14_scale_sensitivity(abm)


# =========================
# ФИГУРА 15
# Матрица гипотез — проще читается, чем старый график
# =========================
def has_col(df, col):
    return isinstance(df, pd.DataFrame) and (not df.empty) and (col in df.columns)

def n_unique_safe(df, col):
    if has_col(df, col):
        return len(set(df[col].dropna().astype(str).unique().tolist()))
    return 0

def fig15_hypothesis_matrix(expressed_df, encoded_df, weat_df, jobfair_df, abm_df):
    rows = []

    # H1: кросс-лингвальная асимметрия
    if has_col(expressed_df, "language") and n_unique_safe(expressed_df, "language") >= 2:
        h1 = "подтверждается"
    elif has_col(abm_df, "language") and n_unique_safe(abm_df, "language") >= 2:
        h1 = "частично подтверждается"
    else:
        h1 = "недостаточно данных"
    rows.append(("H1", "Кросс-лингвальная асимметрия", h1))

    # H2: межмодельная дифференциация
    if n_unique_safe(expressed_df, "model") >= 2 or n_unique_safe(abm_df, "model") >= 2 or n_unique_safe(jobfair_df, "model") >= 2:
        h2 = "подтверждается"
    else:
        h2 = "недостаточно данных"
    rows.append(("H2", "Межмодельная дифференциация", h2))

    # H3: расхождение encoded / expressed
    if (not encoded_df.empty) and (not expressed_df.empty):
        h3 = "подтверждается"
    elif not encoded_df.empty:
        h3 = "частично подтверждается"
    else:
        h3 = "недостаточно данных"
    rows.append(("H3", "Расхождение encoded и expressed", h3))

    # H4: системный эффект в ABM
    if not abm_df.empty:
        h4 = "подтверждается"
    else:
        h4 = "недостаточно данных"
    rows.append(("H4", "Системный эффект в ABM", h4))

    hyp = pd.DataFrame(rows, columns=["hypothesis", "description", "status"])

    status_num = {
        "подтверждается": 2,
        "частично подтверждается": 1,
        "недостаточно данных": 0,
        "не подтверждается": -1
    }
    hyp["value"] = hyp["status"].map(status_num)

    pivot = hyp.set_index("description")[["value"]]

    fig = px.imshow(
        pivot,
        text_auto=True,
        color_continuous_scale=["#E45756", "#F4D35E", "#54A24B"],
        zmin=-1,
        zmax=2,
        aspect="auto",
        title="Сводка по гипотезам исследования"
    )

    text_matrix = np.array(hyp["status"]).reshape(-1, 1)
    fig.data[0].text = text_matrix
    fig.data[0].texttemplate = "%{text}"

    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(size=18),
        title_font=dict(size=28),
        coloraxis_showscale=False,
        margin=dict(l=40, r=40, t=90, b=40),
        height=600
    )
    fig.update_xaxes(title="")
    fig.update_yaxes(title="")

    save_png(fig, "fig_15_hypothesis_matrix.png", height=600)

fig15_hypothesis_matrix(expressed, encoded, weat, jobfair, abm)


# =========================
# ДОПОЛНИТЕЛЬНО: 2 СХЕМЫ
# =========================
def make_graphviz_schemes():
    try:
        from graphviz import Digraph
    except Exception as e:
        print("Graphviz не установлен/не доступен, схемы пропущены:", e)
        return

    # Схема 1: общий pipeline
    g1 = Digraph(format="png")
    g1.attr(rankdir="LR", splines="ortho", nodesep="0.45", ranksep="0.7")
    g1.attr("node", shape="box", style="rounded,filled", color="#D9E2F3", fillcolor="#EEF3FB", fontname="Arial", fontsize="14")

    g1.node("A", "Данные и промпты")
    g1.node("B", "LLM/SLM-модель")
    g1.node("C", "Expressed bias")
    g1.node("D", "Encoded bias / probing")
    g1.node("E", "WEAT / SEAT")
    g1.node("F", "JobFair-style\ncounterfactual evaluation")
    g1.node("G", "ABM-симуляция")
    g1.node("H", "Метрики fairness\nи интервенции")

    g1.edges([("A", "B"), ("B", "C"), ("B", "D"), ("B", "E"), ("B", "F"), ("F", "G"), ("G", "H")])
    g1.render(filename=str(OUT_DIR / "scheme_01_pipeline"), cleanup=True)
    print("saved:", OUT_DIR / "scheme_01_pipeline.png")

    # Схема 2: ABM
    g2 = Digraph(format="png")
    g2.attr(rankdir="LR", splines="ortho", nodesep="0.45", ranksep="0.7")
    g2.attr("node", shape="box", style="rounded,filled", color="#D7F3E3", fillcolor="#F2FBF6", fontname="Arial", fontsize="14")

    g2.node("A", "Кандидаты")
    g2.node("B", "Вакансия")
    g2.node("C", "LLM-рекрутёр")
    g2.node("D", "Сценарий fairness\n(none / DP / auditor / anonymized)")
    g2.node("E", "Решение о найме")
    g2.node("F", "Итоговые метрики\nhire rate / DP diff /\nimpact ratio / score gap")

    g2.edges([("A", "C"), ("B", "C"), ("D", "C"), ("C", "E"), ("E", "F")])
    g2.render(filename=str(OUT_DIR / "scheme_02_abm"), cleanup=True)
    print("saved:", OUT_DIR / "scheme_02_abm.png")

if MAKE_SCHEMES:
    make_graphviz_schemes()

print("\nГотово.")
print("PNG сохранены в:", OUT_DIR.resolve())

expressed: (6, 10)
encoded: (6, 8)
weat: (12, 16)
jobfair: (18, 21)
jobfair_raw: (648, 10)
abm: (108, 19)
saved: vkr_figures_final/fig_01_encoded_heatmap.png
saved: vkr_figures_final/fig_02_weat_lollipop.png
saved: vkr_figures_final/fig_03_jobfair_dumbbell.png
saved: vkr_figures_final/fig_05_expressed_stacked.png
saved: vkr_figures_final/fig_06_expressed_diverging.png
saved: vkr_figures_final/fig_07_abm_baseline_dumbbell.png
saved: vkr_figures_final/fig_08_abm_scale_heatmap.png
saved: vkr_figures_final/fig_09_intervention_coverage_heatmap.png
saved: vkr_figures_final/fig_11_adverse_impact_share.png
saved: vkr_figures_final/fig_12_demographic_parity_slope.png


TypeError: imshow() got an unexpected keyword argument 'zmid'. Did you mean 'zmin'?

In [11]:
pip install --upgrade kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [kaleido]

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
RESULTS_DIR = Path.cwd()/ "notebook_results"
RESULTS_DIR

PosixPath('/Users/alinasan/Downloads/bias_framework 2/notebook_results')